# Naive Bayes — From Scratch

**Module:** Additional Classical Machine Learning  
**Focus:** Multinomial Naive Bayes using NumPy

## Objectives

- Understand Bayes' theorem and conditional probability.
- Derive the Naive Bayes classification rule.
- Understand conditional independence.
- Estimate prior and likelihood probabilities.
- Implement Laplace smoothing.
- Understand log probabilities and numerical stability.
- Evaluate classification performance.
- Compare our implementation with Scikit-learn.

## 1. Mathematical Foundation

Naive Bayes is a supervised learning algorithm based on Bayes' theorem.

Bayes' theorem:

P(C|X) = P(X|C)P(C) / P(X)

Where:

- P(C|X): Posterior probability - prob of class C being true if we have observed X happening
- P(X|C): Likelihood            - prob of X being observed if the class C is true 
- P(C): Prior probability       - belief about class C seeing new data/obs
- P(X): Evidence                - total prob of observing X all across 

### Conditional Independence Assumption

Naive Bayes assumes that features are conditionally independent given the class.

P(X1, X2, ..., Xn | C)
= P(X1|C) * P(X2|C) * ... * P(Xn|C)

Therefore, the classification rule becomes:

C_hat = argmax_C P(C) * product(P(Xi|C))

### Log Probabilities

To avoid numerical underflow, we use logarithms:

log P(C|X) ∝ log P(C) + sum(log P(Xi|C))

For Multinomial Naive Bayes, feature counts are used as weights:

Score(C) = log P(C) + sum(xi * log P(Xi|C))

---
## 2. Dataset Preparation

We will use a small artificial email classification dataset.

Vocabulary:

1. free
2. prize
3. meeting
4. project

Each row represents an email.
Each column represents the frequency of a word.

Class labels:
- 1 = Spam
- 0 = Not Spam

In [7]:
import numpy as np 
X_train = np.array([   #-> feature variable
    [3,2,0,0],
    [2, 1, 0, 0],
    [1, 3, 0, 0],
    [0, 0, 2, 3],
    [0, 0, 1, 2],
    [0, 0, 3, 1]
], dtype=float
)
Y_train = np.array([1,1,1,0,0,0]) #-> target class

feature_names = {"free","prize","meeting","project"}

print("Training shape:", X_train.shape)
print("Labels:", Y_train)



Training shape: (6, 4)
Labels: [1 1 1 0 0 0]


In [8]:
#we will be implementing multinomial naive bias and utilize laplace smoothing
class MultinomialNaiveBayes:

    def __init__(self, alpha=1.0):
        if alpha <= 0:
            raise ValueError("alpha must be positive")
        self.alpha = alpha

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        if X.ndim != 2 or X.shape[0] != len(y):
            raise ValueError("Invalid input dimensions")
        if X.shape[0] == 0 or X.shape[1] == 0:
            raise ValueError("Training data cannot be empty")
        if np.any(X < 0):
            raise ValueError("Features must be non-negative")
        self.classes = np.unique(y)

        n_samples, n_features = X.shape
        n_classes = len(self.classes)
        self.class_log_prior = np.zeros(n_classes)
        self.feature_log_prob = np.zeros(
            (n_classes, n_features)
        )
        for idx, cls in enumerate(self.classes):
            X_class = X[y == cls]
            # Calculate prior probability
            self.class_log_prior[idx] = np.log(
                len(X_class) / n_samples
            )
            # Calculate feature counts
            feature_count = np.sum(X_class, axis=0)
            # Apply Laplace smoothing
            smoothed_count = feature_count + self.alpha
            total_count = np.sum(feature_count)
            # Calculate log likelihood
            self.feature_log_prob[idx] = (
                np.log(smoothed_count)
                - np.log(
                    total_count + self.alpha * n_features
                )
            )
        return self

    def predict_log_proba_scores(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim != 2 or X.shape[1] != self.feature_log_prob.shape[1]:
            raise ValueError("Invalid input dimensions")
        if np.any(X < 0):
            raise ValueError("Features must be non-negative")

        scores = (
            X @ self.feature_log_prob.T
            + self.class_log_prior
        )
        return scores

    def predict(self, X):
        scores = self.predict_log_proba_scores(X)
        best_class_indices = np.argmax(scores, axis=1)
        return self.classes[best_class_indices]

In [9]:
#fitting the model 
model = MultinomialNaiveBayes(alpha=1.0)

model.fit(X_train, Y_train)

print("Classes:", model.classes)

print("Class log priors:")
print(model.class_log_prior)

print("Feature log probabilities:")
print(model.feature_log_prob)

Classes: [0 1]
Class log priors:
[-0.69314718 -0.69314718]
Feature log probabilities:
[[-2.77258872 -2.77258872 -0.82667857 -0.82667857]
 [-0.82667857 -0.82667857 -2.77258872 -2.77258872]]


In [10]:
X_test = np.array([
    [2, 2, 0, 0],
    [0, 0, 2, 2]
], dtype=float)

predictions = model.predict(X_test)

print("Predictions:", predictions)
print("Expected: [1 0]")

assert np.array_equal(predictions, [1, 0])

Predictions: [1 0]
Expected: [1 0]


In [11]:
X_test = np.array([
    [2, 2, 0, 0],
    [0, 0, 2, 2]
], dtype=float)

predictions = model.predict(X_test)

print("Predictions:", predictions)
print("Expected: [1 0]")

assert np.array_equal(predictions, [1, 0])

Predictions: [1 0]
Expected: [1 0]


In [12]:
for alpha in [0.1, 0.5, 1.0, 2.0]:
    smoothed_model = MultinomialNaiveBayes(
        alpha=alpha
    )
    smoothed_model.fit(X_train, Y_train)
    print(f"\nAlpha: {alpha}")
    print("Feature log probabilities:")
    print(np.round(
        smoothed_model.feature_log_prob,
        3
    ))
    print("Predictions:")
    print(smoothed_model.predict(X_test))


Alpha: 0.1
Feature log probabilities:
[[-4.82  -4.82  -0.709 -0.709]
 [-0.709 -0.709 -4.82  -4.82 ]]
Predictions:
[1 0]

Alpha: 0.5
Feature log probabilities:
[[-3.332 -3.332 -0.767 -0.767]
 [-0.767 -0.767 -3.332 -3.332]]
Predictions:
[1 0]

Alpha: 1.0
Feature log probabilities:
[[-2.773 -2.773 -0.827 -0.827]
 [-0.827 -0.827 -2.773 -2.773]]
Predictions:
[1 0]

Alpha: 2.0
Feature log probabilities:
[[-2.303 -2.303 -0.916 -0.916]
 [-0.916 -0.916 -2.303 -2.303]]
Predictions:
[1 0]


In [13]:
def classification_metrics(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (tp + tn) / len(y_true)

    precision = (
        tp / (tp + fp)
        if tp + fp else 0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn else 0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": np.array([
            [tn, fp],
            [fn, tp]
        ])
    }

train_predictions = model.predict(X_train)

metrics = classification_metrics(
    Y_train,
    train_predictions
)

for name, value in metrics.items():
    print(name, ":", value)

accuracy : 1.0
precision : 1.0
recall : 1.0
f1 : 1.0
confusion_matrix : [[3 0]
 [0 3]]
